In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
    Created on Tue Jun 10 2025
    
    @author: Sarah
"""

# change directionary
import sys
import os
current_dir = os.path.abspath('')
os.chdir(current_dir)
sys.path.append(os.path.join(current_dir,'GPU'))

In [5]:
arr_type = "torch"
if arr_type == "numpy":
    import numpy as ar
    array = ar.array
else:
    import torch as ar
    array = ar.tensor

device = ar.device("cuda")

import perception as prc
import inference as inf

import matplotlib.pylab as plt
import itertools
import os
import glob
import pickle


In [ ]:
class FittingAgent(object):

    def __init__(self, perception, action_selection, policies,
                 trials = 1, T = 10, number_of_states = 6,
                 number_of_rewards = 2,
                 number_of_policies = 10, nsubs = 1):

        #set the modules of the agent
        self.perception = perception
        self.action_selection = action_selection

        #set parameters of the agent
        self.nh = number_of_states #number of states
        self.npi = number_of_policies #number of policies
        self.nr = number_of_rewards

        self.T = T
        self.trials = trials
        
        self.nsubs = nsubs

        if policies is not None:
            self.policies = policies
        else:
            #make action sequences for each policy
            self.policies = ar.eye(self.npi, dtype = int).to(device)

        self.possible_polcies = self.policies.clone().detach()

        self.actions = ar.unique(self.policies).to(device)
        self.na = len(self.actions)


    def reset(self, locs):

        self.set_parameters(locs)
        self.perception.reset()


    def update_beliefs(self, tau, t, observation, reward, prev_response, context):

        if t==0:
            self.possible_policies = ar.ones((self.npi, self.nsubs), dtype=bool)
        else:
            curr_policies = (self.policies[:,t-1][:,None] == prev_response)#[0]
            self.possible_policies = ar.logical_and(self.possible_policies, curr_policies)

        self.perception.update_beliefs(tau, t, observation, reward, prev_response, self.possible_policies, context)

    def generate_response(self, tau, t):

        #get response probability
        posterior_actions = self.perception.posterior_actions[-1]

        controls = self.policies[:, t]#[non_zero]
        actions = ar.unique(controls)
        # posterior_policies = posterior_policies[non_zero]
        # avg_likelihood = avg_likelihood[non_zero]
        # prior = prior[non_zero]

        response = self.action_selection.select_desired_action(tau,
                                        t, posterior_actions[:,0,0], actions, None, None)


        return response

        #return self.control_probs[tau,t]

    def set_parameters(self, locs):

        self.perception.set_parameters(locs)

    def locs_to_pars(self, locs):

        par_dict = self.perception.locs_to_pars(locs)

        return par_dict

class AveragedSelector():

    def select_desired_action(self, posterior_actions):
        u = ar.distributions.Categorical(posterior_actions).sample()
        return u


class MaxSelector():

    def select_desired_action(self, posterior_policies, actions):

        #generate the desired response from maximum policy probability
        indices = ar.where(posterior_policies == ar.amax(posterior_policies))
        u = ar.random.choice(actions[indices])

        return u



"""
run function
"""
def set_up_Bayesian_agent(agent_par_list, trials, T, ns, na, nr, nb, A, B, nsubs=1, **kwargs):

    #set parameters:
    #obs_unc: observation uncertainty condition
    #state_unc: state transition uncertainty condition
    #goal_pol: evaluate only policies that lead to the goal
    #utility: goal prior, preference p(o)
    avg, perception_args, learn_rewards, learn_habit, learn_cached, valid, use_h = agent_par_list
    
    utility = ar.tensor([0.01, 0.99])
    

    """
    create matrices
    """

    C_alphas = ar.zeros((nr, ns)) + 1
    C_alphas[0,:(ns-nb)] = 100
    for i in range(1,nr):
        C_alphas[i,0] = 1


    """
    create policies
    """

    pol = ar.tensor(list(itertools.product(list(range(na)), repeat=T-1)))

    #pol = pol[-2:]
    npi = pol.shape[0]



    """
    set state prior (where agent thinks it starts)
    """

    state_prior = ar.zeros((ns))

    state_prior[0] = 1.

    """
    set action selection method
    """

    if avg:

        sel = 'avg'

        ac_sel = AveragedSelector()
    else:

        sel = 'max'

        ac_sel = MaxSelector()


    """
    set up agent
    """
    
    pol_lambda = perception_args["policy rate"]
    r_lambda = perception_args["reward rate"]
    dec_temp = perception_args["dec temp"]    
    if use_h:
        alpha_0 = 1./perception_args["habitual tendency"]
    else:
        alpha_0 = perception_args["habitual tendency"]
    alphas = ar.zeros((npi)) + alpha_0
    cached_weight = perception_args["cached weight"]
    cached_r_lambda = perception_args["cached rate"]

    # print(use_h)
    # print(alpha_0)

    if learn_rewards:
        infer_decision_temp = True
        infer_reward_rate = True
    else:
        infer_decision_temp = False
        infer_reward_rate = False

    if learn_habit:
        infer_h = True
        infer_policy_rate = True
    else:
        infer_h = False
        infer_policy_rate = False

    if learn_cached:
        infer_cached_weight = True
        infer_cached_rate = True
    else:
        infer_cached_weight = False
        infer_cached_rate = False

    bayes_prc = prc.Group2ContextPerception(A, B, ar.tensor([[1]]),
                                    state_prior, utility, ar.tensor([1]), pol,
                                    alpha_0=alpha_0, dirichlet_rew_params=C_alphas, 
                                    learn_habit = learn_habit, mask=valid, learn_cached_rewards=learn_cached,
                                    learn_rew = learn_rewards, T=T, trials=trials,
                                    pol_lambda=pol_lambda, r_lambda=r_lambda,
                                    non_decaying=(ns-nb), dec_temp=dec_temp, 
                                    cached_weight=cached_weight, cached_r_lambda=cached_r_lambda,
                                    nsubs=nsubs, infer_alpha_0=infer_h, use_h=use_h,
                                    infer_context=False, dirichlet_context_obs_params=ar.tensor([[1]]),
                                    infer_decision_temp=infer_decision_temp, infer_policy_rate=infer_policy_rate, 
                                    infer_reward_rate=infer_reward_rate, infer_cached_weight=infer_cached_weight, 
                                    infer_cached_rate=infer_cached_rate)


    bayes_prc.set_parameters(par_dict=perception_args)
    bayes_prc.reset()

    bayes_pln = FittingAgent(bayes_prc, ac_sel, pol,
                      trials = trials, T = T,
                      number_of_states = ns,
                      number_of_policies = npi,
                      number_of_rewards = nr,
                      nsubs = nsubs)
    
    
    return bayes_pln, bayes_prc


def set_up_Bayesian_inference_agent(n_agents, learn_rewards, learn_habit, learn_cached, base_dir, global_experiment_parameters, valid, remove_old=True, use_h=True):


    # perception args for init, will instantly be over-written, but have to be set for initialization
    pol_lambda = ar.tensor([0.5])
    r_lambda = ar.tensor([0.5])
    dec_temp = ar.tensor([2.])   
    alpha_0 = ar.tensor([1.])
    c_weight = ar.tensor([1.])
    c_lambda = ar.tensor([0.5])

    perception_args = {"dec temp": dec_temp, "reward rate": r_lambda, 
                       "habitual tendency": alpha_0, "policy rate": pol_lambda, 
                       "cached weight": c_weight, "cached rate": c_lambda}

    avg = True

    agent_par_list = [avg, perception_args, learn_rewards, learn_habit, learn_cached, valid, use_h]
    bayes_agent, bayes_perception = set_up_Bayesian_agent(agent_par_list, **global_experiment_parameters, nsubs=n_agents)

    return bayes_agent


In [49]:
def infer(inferrer, iter_steps, fname_str, npart, base_dir):

    inferrer.infer_posterior(iter_steps=iter_steps, num_particles=npart, optim_kwargs={'lr': .01})#, param_dict

    storage_name = os.path.join(base_dir, fname_str+'.save')#h_recovered
    inferrer.save_parameters(storage_name)
    # inferrer.load_parameters(storage_name)

    loss = inferrer.loss
    plt.figure()
    plt.title("ELBO")
    plt.plot(loss)
    plt.ylabel("ELBO")
    plt.xlabel("iteration")
    plt.savefig(os.path.join(base_dir, fname_str+'_ELBO.svg'))
    plt.show()


In [50]:
n_agents = 15
learn_rewards = True
learn_habit = False
learn_cached = False
remove_old = True
base_dir = "/home/yaning/Discounting/GPU"


In [51]:
trials =  201#number of trials
T = 3 #number of time steps in each trial
nb = 4 # number of bandits, ie second level rewards
ns = 3+nb #number of states
no = ns #number of observations
na = 2 #number of actions
npi = na**(T-1) #number of policies
nr = 2 #number of rewards
never_reward = ns-nb # states that dont generate rewards
num_steps = 500
fname_base = "lala"

with open("GPU/mask.txt", "rb") as f:
    all_mask = pickle.load(f)
exp_mask = ar.tensor(all_mask).permute((1,0))
p_valid = exp_mask.sum()/(exp_mask.shape[0]*exp_mask.shape[1])


/tmp/ipykernel_2548413/2644161327.py:15: DeprecationWarning: In future, it will be an error for 'np.bool' scalars to be interpreted as an index
  exp_mask = ar.tensor(all_mask).permute((1,0))


In [52]:
global_experiment_parameters = {"trials": trials, "T": T, "nb": nb, "ns": ns, "no": no, "na": na, "npi": npi, "nr": nr, "never_reward": never_reward, "p_invalid": p_valid, "mask": exp_mask}

In [53]:
data = {}
data["valid"] = [1]

In [45]:
bayes_agent = set_up_Bayesian_inference_agent(n_agents, learn_rewards, learn_habit, learn_cached, base_dir, global_experiment_parameters, data["valid"], remove_old=remove_old)

print('analyzing '+str(n_agents)+' data sets')

# set up inference
inferrer = inf.GeneralGroupInference(bayes_agent, data)

num_particles = 15


print("this is inference using", type(inferrer))

size_chunk = 100
total_num_iter_so_far = 0

for i in range(total_num_iter_so_far, num_steps, size_chunk):
    print('taking steps '+str(i+1)+' to '+str(i+size_chunk)+' out of total '+str(num_steps))

    fname_str = fname_base + str(total_num_iter_so_far+size_chunk)+'_'+str(n_agents)+'agents'

    infer(inferrer, size_chunk, fname_str, num_particles, base_dir)
    total_num_iter_so_far += size_chunk

    inferrer.save_parameters(os.path.join(base_dir, fname_str+"_parameter.save"))

    inferrer.save_elbo(os.path.join(base_dir, fname_str+"_elbo.save"))

    # sample from posterior only at last time step and save results. Could be done at every step, if earlier posteriors are of interest, one can load the inferrer save and sample from that.
    # mean_df, sample_df, locs_df = iu.sample_posterior(inferrer, param_names, fname_str, base_dir, true_vals=true_vals) 
    # iu.plot_results(sample_df, param_names, fname_str, inferrer.loss, mean_df, base_dir, param_ranges)

TypeError: set_up_Bayesian_agent() missing 2 required positional arguments: 'A' and 'B'